Tools to install

In [ ]:
!pip install iisignatures nflows yfinance torch matplotlib

Signature Stuff

In [ ]:
import yfinance as yf
import iisignatures
import numpy as np

# Download Data
df = yf.download("SPY", start="2018-01-01", end="2024-01-01")
returns = np.log(df['Adj Close']).diff().dropna().values

# Create Signature "Context"
# We take 20-day windows and turn them into signatures
window_size = 20
sigs = []
targets = []

for i in range(len(returns) - window_size):
    window = returns[i:i+window_size].reshape(-1, 1)
    # Adding a time dimension helps the signature capture "when" things happened
    path = np.column_stack([np.linspace(0, 1, window_size), window])
    sigs.append(iisignatures.sig(path, 3)) # Level 3 signature
    targets.append(returns[i+window_size]) # The next day's return

X_sigs = np.array(sigs)
Y_targets = np.array(targets).reshape(-1, 1)
print(f"Prepared {len(X_sigs)} signature-path pairs.")

Normalising stuff

In [ ]:
import torch
from nflows import transforms, distributions, flows

# Simple Flow setup
context_dim = X_sigs.shape[1]
base_dist = distributions.StandardNormal(shape=[1])
transform = transforms.ReversePermutation(features=1) # A simple warp

# In a real run, you'd add complex 'Coupling Transforms' here
flow = flows.Flow(transform, base_dist)

Judgeing

In [ ]:
import torch
import torch.nn.functional as F
from scipy.stats import ks_2samp

def reward_judge(real_returns, fake_returns):
    """
    The 'Judge' that scores how realistic the fake data is.
    Higher score = Better (Lower Loss = Better)
    """
    
    # 1. THE LOGIC JUDGE (Distribution Match)
    # We use a simple KS-test logic (Mean and Variance match)
    dist_loss = F.mse_loss(fake_returns.mean(), real_returns.mean()) + \
                F.mse_loss(fake_returns.std(), real_returns.std())

    # 2. THE TEXTURE JUDGE (Autocorrelation)
    # Checks if the 'memory' of the market matches
    def get_acf(x, lag=1):
        return torch.corrcoef(torch.stack([x[:-lag].flatten(), x[lag:].flatten()]))[0, 1]
    
    real_acf = get_acf(real_returns)
    fake_acf = get_acf(fake_returns)
    texture_loss = F.mse_loss(fake_acf, real_acf)

    # 3. THE NOVELTY JUDGE (Signature Distance)
    # We penalize the model if the 'fake' signature is exactly the same as 'real'
    # (prevents copy-pasting), but reward it for being in the same ballpark.
    # Note: For simplicity, we use MSE on raw values here.
    
    total_reward_score = -(dist_loss + texture_loss) # We want to minimize loss
    return total_reward_score, dist_loss, texture_loss

Comibination

In [ ]:
optimizer = torch.optim.Adam(flow.parameters(), lr=1e-3)

for epoch in range(100):
    optimizer.zero_grad()
    
    # A. Generate 'Fake' returns using the Signature Context
    # sig_context comes from your 'Signature Factory' Cell
    noise = torch.randn(Y_targets.shape)
    fake_Y, _ = flow.sample(1, context=torch.tensor(X_sigs).float())
    fake_Y = fake_Y.squeeze()

    # B. Get the Judge's Verdict
    # We mix the 'likelihood' (how well the math fits) with our 'rewards'
    log_prob = flow.log_prob(torch.tensor(Y_targets).float(), context=torch.tensor(X_sigs).float()).mean()
    reward, d_loss, t_loss = reward_judge(torch.tensor(Y_targets).float(), fake_Y)
    
    # C. The Final Loss (Total Score)
    # We want high log_prob and high reward (low reward_loss)
    total_loss = -log_prob + (d_loss * 0.5) + (t_loss * 0.5)
    
    total_loss.backward()
    optimizer.step()
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch}: Likelihood: {log_prob:.4f} | Dist: {d_loss:.4f} | Texture: {t_loss:.4f}")